# 🏥 Apollo Voice Engine - Phase 2: Training

**Modality Alignment + LoRA Fine-tuning**

This notebook trains the model to understand audio tokens by:
1. **Stage 1**: Alignment training (Audio ↔ Text mapping)
2. **Stage 2**: LoRA fine-tuning on medical dialogues

---

⚠️ **GPU Required**: Runtime → Change runtime type → **T4 GPU** (or A100 for faster training)

## 1️⃣ Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate snac sentencepiece torchaudio
!pip install -q datasets peft bitsandbytes wandb

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2️⃣ Load IndicVoices Dataset (Sample)

In [ ]:
from datasets import load_dataset

# Load a small sample for demo (Hindi)
# Full training would use all languages: hi, ta, te, kn
print("Loading IndicVoices Hindi sample...")

try:
    dataset = load_dataset(
        "ai4bharat/IndicVoices",
        "hi",  # Hindi
        split="train",
        streaming=True,
        trust_remote_code=True
    )
    
    # Take first 100 samples for demo
    samples = list(dataset.take(100))
    print(f"✓ Loaded {len(samples)} samples")
    
    # Show sample structure
    if samples:
        print(f"\nSample keys: {samples[0].keys()}")
        print(f"Transcription: {samples[0].get('transcription', 'N/A')[:100]}...")
        
except Exception as e:
    print(f"Note: IndicVoices requires authentication. Error: {e}")
    print("\nUsing synthetic data for demo instead...")
    samples = None

## 3️⃣ Create Synthetic Training Data

For demo purposes, we'll create synthetic audio-text pairs.
In production, use real IndicVoices data.

In [ ]:
import numpy as np

# Medical dialogue examples for training
MEDICAL_DIALOGUES = [
    # Hindi
    {"lang": "hi", "query": "मुझे सिरदर्द हो रहा है", "response": "कृपया बताइए कि सिरदर्द कब से है और कितना तेज़ है।"},
    {"lang": "hi", "query": "कार्डियोलॉजी विभाग कहाँ है?", "response": "कार्डियोलॉजी विभाग दूसरी मंजिल पर है, लिफ्ट के बाएं तरफ।"},
    {"lang": "hi", "query": "मेरा अपॉइंटमेंट कब है?", "response": "कृपया अपना नाम और फोन नंबर बताइए, मैं चेक करता हूँ।"},
    
    # Tamil
    {"lang": "ta", "query": "எனக்கு தலைவலி", "response": "தலைவலி எப்போது தொடங்கியது என்று சொல்லுங்கள்."},
    {"lang": "ta", "query": "மருந்து கடை எங்கே?", "response": "மருந்து கடை தரை தளத்தில் உள்ளது."},
    
    # Telugu  
    {"lang": "te", "query": "నాకు జ్వరం వచ్చింది", "response": "జ్వరం ఎంత ఉంది? థర్మామీటర్‌తో చెక్ చేశారా?"},
    {"lang": "te", "query": "డాక్టర్ ఎప్పుడు వస్తారు?", "response": "డాక్టర్ సాయంత్రం 4 గంటలకు వస్తారు."},
    
    # Kannada
    {"lang": "kn", "query": "ನನಗೆ ಜ್ವರ ಬಂದಿದೆ", "response": "ಜ್ವರ ಎಷ್ಟು ಇದೆ? ಥರ್ಮಾಮೀಟರ್‌ನಲ್ಲಿ ಚೆಕ್ ಮಾಡಿದ್ದೀರಾ?"},
    {"lang": "kn", "query": "ಆಂಬುಲೆನ್ಸ್ ಬೇಕು", "response": "ಆಂಬುಲೆನ್ಸ್ ಕರೆಯುತ್ತೇನೆ. ನಿಮ್ಮ ವಿಳಾಸ ಹೇಳಿ."},
]

print(f"Created {len(MEDICAL_DIALOGUES)} medical dialogue pairs")
print(f"Languages: {set(d['lang'] for d in MEDICAL_DIALOGUES)}")

## 4️⃣ Load Models for Training

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC
from peft import LoraConfig, get_peft_model, TaskType

# Constants
AUDIO_VOCAB_SIZE = 4096
AUDIO_START = "<|audio_start|>"
AUDIO_END = "<|audio_end|>"

# Load SNAC
print("Loading SNAC...")
snac = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda").eval()

# Load tokenizer
print("Loading Sarvam-1 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("sarvamai/sarvam-1", trust_remote_code=True)
tokenizer.add_special_tokens({"additional_special_tokens": [AUDIO_START, AUDIO_END]})

# Load model with 4-bit quantization for memory efficiency
print("Loading Sarvam-1 with 4-bit quantization...")
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    "sarvamai/sarvam-1",
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto"
)

# Extend vocabulary
original_vocab = model.config.vocab_size
extended_vocab = original_vocab + 2 + AUDIO_VOCAB_SIZE
model.resize_token_embeddings(extended_vocab)

print(f"✓ Vocab: {original_vocab} → {extended_vocab}")
print(f"✓ Model loaded with 4-bit quantization")

## 5️⃣ Configure LoRA for Efficient Training

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=64,  # Rank
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n✓ LoRA applied - only ~1% of parameters are trainable!")

## 6️⃣ Training Data Preparation

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MedicalDialogueDataset(Dataset):
    """Dataset for medical dialogues with simulated audio tokens."""
    
    def __init__(self, dialogues, tokenizer, max_length=512):
        self.dialogues = dialogues
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.dialogues)
    
    def __getitem__(self, idx):
        d = self.dialogues[idx]
        
        # Format: [Query] {AUDIO_START} {simulated_audio_tokens} {AUDIO_END} [Response]
        # In real training, audio_tokens come from SNAC encoding
        
        prompt = f"Patient Query: {d['query']}\nAssistant Response: {d['response']}"
        
        encoded = self.tokenizer(
            prompt,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "labels": encoded["input_ids"].squeeze()  # Causal LM uses same as labels
        }

# Create dataset
train_dataset = MedicalDialogueDataset(MEDICAL_DIALOGUES, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Batches: {len(train_loader)}")

## 7️⃣ Training Loop

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

# Training config
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Training
model.train()
total_loss = 0

print(f"Starting training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in progress:
        optimizer.zero_grad()
        
        # Move to GPU
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        labels = batch["labels"].to("cuda")
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        epoch_loss += loss.item()
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        progress.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

print("\n✓ Training complete!")

## 8️⃣ Test the Trained Model

In [ ]:
import time

model.eval()

test_queries = [
    "Patient Query: मुझे छाती में दर्द हो रहा है\nAssistant Response:",
    "Patient Query: கார்டியாலஜி எங்கே?\nAssistant Response:",
    "Patient Query: నా అపాయింట్‌మెంట్ ఏమిటి?\nAssistant Response:",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Input: {query}")
    
    inputs = tokenizer(query, return_tensors="pt").to("cuda")
    
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = (time.time() - start) * 1000
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Output: {response}")
    print(f"⏱ Latency: {latency:.0f}ms")

## 9️⃣ Save Fine-tuned Model

In [ ]:
# Save LoRA weights
output_dir = "apollo_voice_lora"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✓ Model saved to {output_dir}/")

# Download to local machine
from google.colab import files
import shutil

# Zip the model
shutil.make_archive("apollo_voice_lora", 'zip', output_dir)
print("\n📥 Click below to download the trained model:")
files.download("apollo_voice_lora.zip")

---

## 📊 Training Summary

| Metric | Value |
|--------|-------|
| Base Model | Sarvam-1 2B |
| Training Method | LoRA (r=64) |
| Trainable Params | ~1% |
| Languages | Hindi, Tamil, Telugu, Kannada |
| Quantization | 4-bit (NF4) |

### Next Steps:
1. **Scale up training** with full IndicVoices dataset
2. **Add audio tokens** by encoding real speech with SNAC
3. **TensorRT-LLM optimization** for <300ms latency
4. **Deploy to Apollo kiosks** with WebRTC interface